# Dive Deeper - Regression Models

## Study Case: Prediction of Gas Production on Volve Field Production Data

In [36]:
# import library 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import sklearn
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn import svm

## Task 1 : Load Data and Exploratory Data Analysis

Baca data `volve_field_gas.csv` dan simpanlah kedalam variabel dengan nama `volve`

In [37]:
volve = pd.read_csv('data_input/volve_field_gas.csv')
volve.head()

,DATE_PERIOD,NPD_WELL_BORE_NAME,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,AVG_DOWNHOLE_TEMPERATURE,AVG_DP_TUBING,AVG_ANNULUS_PRESS,AVG_CHOKE_SIZE_P,AVG_WHP_P,AVG_WHT_P,BORE_GAS_VOL
0,2008-03-21,15/9-F-12,24.0,274.072,105.916,174.498,6.696,32.08725,99.574,75.356,458447.0
1,2008-03-22,15/9-F-12,24.0,273.456,105.915,174.284,6.462,32.07839,99.172,75.319,458028.0
2,2008-03-23,15/9-F-12,24.0,272.881,105.913,174.088,6.288,32.06070,98.793,75.353,466335.0
3,2008-03-24,15/9-F-12,24.0,272.237,105.913,173.835,4.957,32.12839,98.402,74.812,468480.0
4,2008-03-25,15/9-F-12,24.0,271.684,105.910,173.644,14.906,32.04210,98.039,74.959,406562.0


Berikut ini adalah deskripsi dan penjelasan pada setiap kolomnya:  

- `DATE_PERIOD`: Tanggal produksi.  
- `NPD_WELL_BORE_NAME`: Nama dari sumur bor.  
- `ON_STREAM_HRS`: Durasi produksi secara aktif pada hari tersebut.  
- `AVG_DOWNHOLE_PRESSURE`: Tekanan rata-rata (bar) di dalam sumur pada lokasi bawah permukaan.  
- `AVG_DOWNHOLE_TEMPERATURE`: Suhu rata-rata (°C) di dalam sumur pada lokasi bawah permukaan.  
- `AVG_DP_TUBING`: Perbedaan tekanan rata-rata (bar) pada bagian tubing.  
- `AVG_ANNULUS_PRESS`: Tekanan rata-rata (bar) pada bagian annulus, yaitu ruang antara casing dan tubing pada sumur.  
- `AVG_CHOKE_SIZE_P`: Pengaturan rata-rata ukuran katup choke (dalam persen), yang berfungsi mengatur aliran fluida dari sumur.  
- `AVG_WHP_P`: Pengaturan rata-rata tekanan kepala sumur (dalam persen), yang mengontrol tekanan pada permukaan kepala sumur.  
- `AVG_WHT_P`: Pengaturan rata-rata suhu kepala sumur (dalam persen), yang mengontrol suhu pada permukaan kepala sumur.  
- `BORE_GAS_VOL`: Volume gas bumi (cubic feet) yang diproduksi.

### Tipe Data, Missing Value dan Duplikat

#### Tipe Data

Coba check tipe data untuk setiap kolom dari data `volve`. Apakah tipe data telah sesuai?

In [38]:
# code here 
volve.dtypes

DATE_PERIOD                  object
NPD_WELL_BORE_NAME           object
ON_STREAM_HRS               float64
AVG_DOWNHOLE_PRESSURE       float64
AVG_DOWNHOLE_TEMPERATURE    float64
AVG_DP_TUBING               float64
AVG_ANNULUS_PRESS           float64
AVG_CHOKE_SIZE_P            float64
AVG_WHP_P                   float64
AVG_WHT_P                   float64
BORE_GAS_VOL                float64
dtype: object

### Missing Value

Mari kita melihat apakah terdapat missing value terhadap data kita berhubung missing value dapat menggagalkan proses pemodelan machine learning.

In [39]:
# code here
volve.isnull().sum()

DATE_PERIOD                 0
NPD_WELL_BORE_NAME          0
ON_STREAM_HRS               0
AVG_DOWNHOLE_PRESSURE       0
AVG_DOWNHOLE_TEMPERATURE    0
AVG_DP_TUBING               0
AVG_ANNULUS_PRESS           0
AVG_CHOKE_SIZE_P            0
AVG_WHP_P                   0
AVG_WHT_P                   0
BORE_GAS_VOL                0
dtype: int64

In [ ]:
volve['']

## Task 2 - Data Preprocessing

### Feature Selection

Tentukan kolom-kolom yang akan digunakan sebagai variabel target dan prediktor

In [40]:
# code here
y = volve['BORE_GAS_VOL']
x = volve.drop(columns=['BORE_GAS_VOL','DATE_PERIOD'],axis=1)

### Categorical Encoding

Lakukan proses Categorical Encoding untuk mengubah data category menjadi numerik

In [41]:
# code here
x = pd.get_dummies(data=x,columns=['NPD_WELL_BORE_NAME'], drop_first=True)
x

,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,AVG_DOWNHOLE_TEMPERATURE,AVG_DP_TUBING,AVG_ANNULUS_PRESS,AVG_CHOKE_SIZE_P,AVG_WHP_P,AVG_WHT_P,NPD_WELL_BORE_NAME_15/9-F-12,NPD_WELL_BORE_NAME_15/9-F-14,NPD_WELL_BORE_NAME_15/9-F-15
0,24.00,274.072,105.916,174.498,6.696,32.08725,99.574,75.356,True,False,False
1,24.00,273.456,105.915,174.284,6.462,32.07839,99.172,75.319,True,False,False
2,24.00,272.881,105.913,174.088,6.288,32.06070,98.793,75.353,True,False,False
3,24.00,272.237,105.913,173.835,4.957,32.12839,98.402,74.812,True,False,False
4,24.00,271.684,105.910,173.644,14.906,32.04210,98.039,74.959,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...
4103,24.00,253.399,105.267,226.401,22.564,100.00000,26.999,83.821,False,False,False
4104,24.00,253.210,105.272,226.255,22.720,100.00000,26.955,84.780,False,False,False
4105,24.00,252.939,105.274,226.037,22.581,100.00000,26.902,82.589,False,False,False
4106,24.00,252.892,105.276,226.022,22.571,100.00000,26.870,83.608,False,False,False


### Train-Test Splitting

Untuk melakukan evaluasi, kita perlu melakukan *Train-Test Splitting* dengan membagi data kita menjadi data train dan data test, dimana:

- Data `train`: Data yang model gunakan untuk training.

- Data `test`: Data untuk evaluasi model

🔻Lakukan **train-test splitting** untuk data train dan test menggunakan `x` dan `y` dengan fungsi `train_test_split()` dan gunakan `random_state=1` serta `test_size=0.2`

In [43]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=1)

## Task 3 - Model Fitting and Evaluation

### Model Fitting and Improvements

Buat model machine learning untuk mempelajari data `volve` serta lakukan proses hyperparameter tuning untuk mengimprove performa model

In [45]:
# code here
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
scaler = StandardScaler()
model = svm.SVR()
model.fit(x_train, y_train)
y_pred = model.predict(x_test)
mae = mean_absolute_error(y_test, y_pred)
print('MAE Support Vector Regressor:', mae)

MAE Support Vector Regressor: 140039.77618070852


In [46]:
# code here
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
model = RandomForestRegressor()
model.fit(x_train, y_train)
y_pred = model.predict(x_test)
mae = mean_absolute_error(y_test, y_pred)
print('MAE Random Forest Regressor:', mae)

MAE Random Forest Regressor: 8017.341872871047


In [48]:
# code here
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error
model = DecisionTreeRegressor()
model.fit(x_train, y_train)
y_pred = model.predict(x_test)
mae = mean_absolute_error(y_test, y_pred)
print('MAE Random Forest Regressor:', mae)

MAE Random Forest Regressor: 10187.946472019465


In [49]:
# code here
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
scaler = StandardScaler()
x_train_scale = scaler.fit_transform(x_train)
x_test_scale = scaler.transform(x_test)
model = svm.SVR()
model.fit(x_train_scale, y_train)
y_pred = model.predict(x_test_scale)
mae = mean_absolute_error(y_test, y_pred)
print('MAE Support Vector Regressor:', mae)

MAE Support Vector Regressor: 139783.41752818334


In [52]:
# code here
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
scaler = StandardScaler()
x_train_scale = scaler.fit_transform(x_train)
x_test_scale = scaler.transform(x_test)
model = xgb.XGBRegressor()
model.fit(x_train_scale, y_train)
y_pred = model.predict(x_test_scale)
mae = mean_absolute_error(y_test, y_pred)
print('MAE Support Vector Regressor:', mae)

MAE Support Vector Regressor: 9151.45775283108


In [53]:
# code here
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
scaler = StandardScaler()
x_train_scale = scaler.fit_transform(x_train)
x_test_scale = scaler.transform(x_test)
model = lgb.LGBMRegressor()
model.fit(x_train_scale, y_train)
y_pred = model.predict(x_test_scale)
mae = mean_absolute_error(y_test, y_pred)
print('MAE LightGBM Regressor:', mae)

[LightGBM] [Warning] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001362 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1825
[LightGBM] [Info] Number of data points in the train set: 3286, number of used features: 11
[LightGBM] [Info] Start training from score 183770.261716
MAE LightGBM Regressor: 9621.079316736203


### Model Prediction

Lakukan prediksi menggunakan model machine learning yang sudah dibangun `model` terhadap data test (`y_pred`)

### Model Evaluation

🔻 Gunakan `mean_absolute_error()` dari library `sklearn` untuk menghitung error performa model:

In [23]:
# code here
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(y_test, y_pred)
print('MAE:', mae)



MAE: 8632.851333333334
